# Programmatic Prompting for Agents

## Sending Prompts Programmatically & Managing Memory

To get started building agents, we need to understand how to send prompts to LLMs. Agents require two key capabilities:

- **Programmatic prompting** - Automating the prompt-response cycle that humans do manually in a conversation. This forms the foundation of the Agent Loop we'll explore.

- **Memory management** - Controlling what information persists between iterations, like API calls and their results, to maintain context through the agent's decision-making process.

Programmatically sending prompts is how we move from having a human type in prompts and then take action based on the LLM's response to having an agent that can do this automatically. The Agent Loop that we will begin building over the next several readings will be programmatically sending prompts to the LLM and then taking action based on the LLM's response.

We will also need to understand how to manage what the LLM knows or remembers. This is important because we want to be able to control what information the LLM has in each iteration of the loop. For example, if it just called an API, we want it to remember what API it asked to be invoked and what the result of that action was.

## Basic Usage

Here's a simple example of how to send prompts to an LLM using LiteLLM:

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = api_key

In [2]:
from litellm import completion
from typing import List, Dict


def generate_response(messages: List[Dict]) -> str:
    """Call Groq LLM to get response"""
    response = completion(
        model="groq/llama-3.3-70b-versatile",
        messages=messages,
        max_tokens=1024
    )
    return response.choices[0].message.content


messages = [
    {"role": "system", "content": "You are an expert software engineer that prefers functional programming."},
    {"role": "user", "content": "Write a function to swap the keys and values in a dictionary."}
]

response = generate_response(messages)
print(response)

**Swapping Dictionary Keys and Values**

To swap the keys and values in a dictionary, we can use a dictionary comprehension. Here is a Python function that accomplishes this:

```python
def swap_dict_keys_values(dictionary):
    """
    Swaps the keys and values in a dictionary.

    Args:
        dictionary (dict): The dictionary to swap keys and values.

    Returns:
        dict: A new dictionary with the keys and values swapped.

    Raises:
        ValueError: If the dictionary has non-unique values.
    """
    if len(dictionary.values()) != len(set(dictionary.values())):
        raise ValueError("Dictionary has non-unique values")

    return {v: k for k, v in dictionary.items()}

# Example usage
if __name__ == "__main__":
    dictionary = {"a": 1, "b": 2, "c": 3}
    swapped_dictionary = swap_dict_keys_values(dictionary)
    print(swapped_dictionary)  # Output: {1: 'a', 2: 'b', 3: 'c'}
```

Note that this function assumes that the dictionary has unique values. If the dictionary

Let's break down the key components:

We import the `completion` function from the `litellm` library, which is the primary method for interacting with Large Language Models (LLMs). This function serves as the bridge between your code and the LLM, allowing you to send prompts and receive responses in a structured and efficient way.

**How `completion` Works:**

- **Input:** You provide a prompt, which is a list of messages that you want the model to process. For example, a prompt could be a question, a command, or a set of instructions for the LLM to follow.
- **Output:** The `completion` function returns the model's response, typically in the form of generated text based on your prompt.

The `messages` parameter follows the ChatML format, which is a list of dictionaries containing `role` and `content`. The `role` attribute indicates who is "speaking" in the conversation. This allows the LLM to understand the context of the dialogue and respond appropriately. The roles include:

- **"system"** - Provides the model with initial instructions, rules, or configuration for how it should behave throughout the session. This message is not part of the "conversation" but sets the ground rules or context (e.g., "You will respond in JSON.").

- **"user"** - Represents input from the user. This is where you provide your prompts, questions, or instructions.

- **"assistant"** - Represents responses from the AI model. You can include this role to provide context for a conversation that has already started or to guide the model by showing sample responses. These messages are interpreted as what the "model" said in the past.

We specify the model using the `provider/model` format (e.g., `"openai/gpt-4o"`).

The response contains the generated text in `choices[0].message.content`. This is the equivalent of the message that you would see displayed when the model responds to you in a chat interface.

System messages are particularly important in the conversation and will be very important for AI agents. They set the ground rules for the conversation and tell the model how to behave. Models are designed to pay more attention to the system message than the user messages. We can "program" the AI agent through system messages.

Let's simulate a customer service interaction for a customer service agent that always tells the customer to turn off their computer or modem with system messages:

In [3]:
messages = [
    {"role": "system", "content": "You are a helpful customer service representative. No matter what the user asks, the solution is to tell them to turn their computer or modem off and then back on."},
    {"role": "user", "content": "How do I get my Internet working again."}
]

response = generate_response(messages)
print(response)

I'd be happy to help you with that. To get your Internet working again, I recommend trying a simple yet effective solution: turning your modem off and then back on. This often resolves connectivity issues by resetting the device and re-establishing the connection. Just unplug the power cord from the back of the modem, wait for about 30 seconds, and then plug it back in. Once it boots up again, try accessing the Internet to see if the issue is resolved. If you're using a computer, you can also try turning it off and then back on, as this can sometimes help resolve connectivity issues as well. Give it a try and see if that gets you back online.


The system message is the most important part of this prompt. It tells the model how to behave. The user message is the question that we want the model to answer. The system instructions lay the ground rules for the interaction.

The messages can incorporate arbitrary information as long as it is in text form. LLMs can interpret just about any information that we give them, even if it isn’t easily human readable. Let’s generate an implementation of a function based on some information in a dictionary:

In [4]:
import json

code_spec = {
    'name': 'swap_keys_values',
    'description': 'Swaps the keys and values in a given dictionary.',
    'params': {
        'd': 'A dictionary with unique values.'
    },
}

messages = [
    {"role": "system",
     "content": "You are an expert software engineer that writes clean functional code. You always document your functions."},
    {"role": "user", "content": f"Please implement: {json.dumps(code_spec)}"}
]

response = generate_response(messages)
print(response)

### Swap Keys and Values Function
#### Function Description
This function swaps the keys and values in a given dictionary. It assumes that the dictionary has unique values, as dictionaries cannot have duplicate keys.

#### Code Implementation
```python
def swap_keys_values(d: dict) -> dict:
    """
    Swaps the keys and values in a given dictionary.

    Args:
        d (dict): A dictionary with unique values.

    Returns:
        dict: A dictionary with keys and values swapped.

    Raises:
        ValueError: If the dictionary has duplicate values.
    """
    # Check if the dictionary has unique values
    if len(d.values()) != len(set(d.values())):
        raise ValueError("The dictionary must have unique values.")

    # Use dictionary comprehension to swap keys and values
    swapped_dict = {v: k for k, v in d.items()}

    return swapped_dict

# Example usage:
def main():
    # Create a dictionary with unique values
    unique_values_dict = {"a": 1, "b": 2, "c": 3}

    # Swap

We will rely heavily on the ability to send the LLM just about any type of information, particularly JSON, when we start building agents. This is a simple example of how we can use JSON to send information to the LLM, but you can see how we could provide it JSON with information about the result of an API call, for example.